In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost
import optuna

from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score,roc_curve,f1_score
from xgboost import XGBClassifier,callback

In [2]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

In [3]:
test_id = test['id']

In [4]:
train.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
train_cols = train.select_dtypes(['object']).columns
test_cols = test.select_dtypes(['object']).columns

In [6]:
train = pd.get_dummies(data=train,columns=train_cols[:-1],drop_first=True,dtype=int)
test = pd.get_dummies(data=test,columns=test_cols,drop_first=True,dtype=int)

In [7]:
train.drop(columns=['id'],inplace = True)
test.drop(columns=['id'],inplace = True)

In [8]:
train['Churn'] = train['Churn'].map({'No':0,'Yes':1})

In [9]:
X = train.drop(columns=['Churn'])
y = train['Churn']

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1,random_state=42,stratify=y)

In [11]:
model = XGBClassifier(
    learning_rate=0.02,
    tree_method='hist',
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.4,
    n_estimators=3300,
    max_depth=5,
    gamma=0.3,
    min_child_weight=6,
    device='cuda',
    early_stopping_rounds=150,
    reg_alpha=3
)

In [12]:
oof_preds = np.zeros(len(X)) 
# test_preds_total: To store the average predictions for the test set (for submission)
test_preds_total = np.zeros(len(test))
# Initialize the S-KFold strategy
n_s = 12
skf = StratifiedKFold(n_splits=n_s, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=0
    )
    
    pre_y_xgb = model.predict_proba(X_val)[:, 1]
    print(f'Score of Fold {fold} XGB: {roc_auc_score(y_val,pre_y_xgb):.5f} %')

    

    test_preds_total += model.predict_proba(test)[:, 1]/n_s
    oof_preds[val_idx] = pre_y_xgb
    print(f"Fold {fold} finished.")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [05:04:24] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Score of Fold 0 XGB: 0.91706 %
Fold 0 finished.
Score of Fold 1 XGB: 0.91629 %
Fold 1 finished.
Score of Fold 2 XGB: 0.91861 %
Fold 2 finished.
Score of Fold 3 XGB: 0.91411 %
Fold 3 finished.
Score of Fold 4 XGB: 0.91790 %
Fold 4 finished.
Score of Fold 5 XGB: 0.91628 %
Fold 5 finished.
Score of Fold 6 XGB: 0.91670 %
Fold 6 finished.
Score of Fold 7 XGB: 0.91855 %
Fold 7 finished.
Score of Fold 8 XGB: 0.91630 %
Fold 8 finished.
Score of Fold 9 XGB: 0.91794 %
Fold 9 finished.
Score of Fold 10 XGB: 0.91623 %
Fold 10 finished.
Score of Fold 11 XGB: 0.91360 %
Fold 11 finished.


In [13]:
print(f'XGB Train score:{roc_auc_score(y,oof_preds):.5f}')

XGB Train score:0.91663


In [14]:
preditions = test_preds_total

In [15]:
df = pd.DataFrame({
    'id':test_id,
    'Churn':preditions
})

In [16]:
df.head()

,id,Churn
0,594194,0.075013
1,594195,0.000814
2,594196,0.118530
3,594197,0.003118
4,594198,0.522798


In [17]:
df.to_csv('sub_3.csv',index=False)